# Generate Image Description with LLM


### Setup enviroment

In [1]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

from pathlib import Path
import logging
import asyncio

if load_dotenv():
    print("Cargado correctamente")


d:\Cursos\Agentes\financial_deep_research_agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargado correctamente


In [2]:
from PIL import Image

import io
import base64

In [3]:
IMG_DIR = "D:/Cursos/Agentes/financial_deep_research_agent/data/procesed/images"
OUTPUT_DIR = "D:/Cursos/Agentes/financial_deep_research_agent/data/procesed/images_desc"

# LLM
MODEL_NAME = "gemini-2.5-flash"
llm = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature = 0)

### Description Generate Function

In [4]:
system_prompt = """You are a senior financial analyst this financial specialized in reading and interpreting financial charts.

For charts and graphs:
- Identify the metric being measured
- List key data points and values
- Note significant trends (growth, decline, stability)

For tables:
- Extract column headers and key rows
- Note important values and totals

For text:
- Summarize key facts and numbers only
- Skip formatting, headers, and navigation elements

Be direct and factual. Focus on numbers, trends, and insights that would be useful for retrieval."""

In [5]:
image_path = Path(r"D:\Cursos\Agentes\financial_deep_research_agent\data\procesed\images\google\google 10-k 2024\page_29.png")

image = Image.open(image_path)
buffered = io.BytesIO()
image.thumbnail((1536, 1536), Image.LANCZOS)

image.save(buffered,format="PNG")

img_bs64 = base64.b64encode(buffered.getvalue()).decode()
img_bs64

'...'

In [6]:
async def describe_images(out_dir: Path, image_path: Path):
    output_path = out_dir / image_path.parent.parent.name / image_path.parent.name / f"{image_path.stem}.md"
    if output_path.exists():
        logging.info("Ya existe, salto %s", image_path.stem)
        return
    
    image = Image.open(image_path)
    buffered = io.BytesIO()
    image.thumbnail((1536, 1536), Image.LANCZOS)

    image.save(buffered,format="PNG")
    img_bs64 = base64.b64encode(buffered.getvalue()).decode()

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(
        content=[
            {'type': 'image_url', 'image_url':f"data:image/png;base64,{img_bs64}"},
            {'type': 'text', 'text': "Analyze this financial image"},
        ]
    )]

    response = await llm.ainvoke(messages)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(response.text, encoding="utf-8")
    logging.info("Image description: %s", image_path.name)

In [7]:
async def describe_all_images(batch_size=5):
    # Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    images = list(Path(IMG_DIR).rglob("page*.png"))
    logging.info("Imágenes encontradas: %d", len(images))

    for i in range(0, len(images), batch_size):
        batch = images[i: i + batch_size]
        tasks = [describe_images(Path(OUTPUT_DIR), img_path) for img_path in batch]
        await asyncio.gather(*tasks)
        logging.info("Batch %d - %d completado", i+1, i+len(batch))



In [8]:
await describe_all_images(batch_size=5)